In [357]:
# Use inline backend for static figures in VS Code
%matplotlib qt

import numpy as np
from tqdm import tqdm
import networkx as nx
import buildgraph
import scipy.signal
import collections

rng = np.random.default_rng(1234)

import matplotlib.pyplot as plt
#import matplotlib.funcanimation as animation

print('Matplotlib backend:', plt.get_backend())

from matplotlib.animation import FuncAnimation, FFMpegWriter
from IPython.display import HTML

Matplotlib backend: QtAgg


Hénon model: 
\begin{align}
    x_{n+1} &= y_n + 1 - ax_n^2\\
    y_{n+1} &= bx_n
\end{align}


In [330]:
def henon_map(a, b, x0, y0, n):
    """
    Generate points of the Henon map
    """
    x, y = x0, y0
    points = np.zeros((n, 2))
    for i in range(n):
        x_new = 1 - a * x**2 + y
        y_new = b * x
        points[i, :] = [x_new, y_new]
        x, y = x_new, y_new
    return points


def plot_henon_map(a, b, x0, y0, n, transitoire):
    """
    Plot the Henon map
    """
    points = henon_map(a, b, x0, y0, n)
    fig, ax = plt.subplots()
    ax.scatter(points[transitoire:, 0], points[transitoire:, 1])
    ax.set_title(f"Henon Map (a={a}, b={b})")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.axis("equal")
    ax.grid(True)
    plt.show()


def plot_bifurcation_diagram(a_values, b, x0, y0, n, last, axis):
    """
    Plot the bifurcation diagram of the Henon map
    """
    fig, ax = plt.subplots()
    if axis == "x":
        for a in a_values:
            points = henon_map(a, b, x0, y0, n)
            ax.scatter([a] * last, points[-last:, 0], s=0.5, c="blue")
    else:
        for a in a_values:
            points = henon_map(a, b, x0, y0, n)
            ax.scatter([a] * last, points[-last:, 1], s=0.5, c="blue")
    ax.set_title(f"Bifurcation Diagram (b={b})")
    ax.set_xlabel("a")
    ax.set_ylabel("x")
    ax.grid(True)
    plt.show()

In [329]:
plot_henon_map(a=1.2, b=0.3, x0=0.1, y0=0.1, n=100000, transitoire=100)

In [298]:
plot_bifurcation_diagram(a_values=np.linspace(0, 1.6, 1000), b=0.3, x0=0.1, y0=0.1, n=10000, last=100, axis="x")

/var/folders/bq/1msz2xh13vbbx9987xtkv7m00000gn/T/ipykernel_10387/3436512866.py:8: RuntimeWarning: overflow encountered in scalar power
  x_new = 1 - a * x**2 + y


D'abord quelques fonctions pour transformer un graphe en Matrice de conectivité

In [299]:
def connectivty(G): 
    return nx.to_numpy_array(G)

#nx.to_scipy_sparse_array( ,format='csr')

def neighbors(A, i):
    neighborhood = []
    for j in range(len(A)):
        if A[j, i] == 1.0:
            neighborhood.append(j)
    return neighborhood

les fonctions pour l'intégration

In [300]:
def transfo_coupling_vec(G, X, Y, Eps):
    # X_new = np.zeros(len(G.nodes))
    # Y_new = np.zeros(len(G.nodes))
    m, n = len(X), len(Y)
    assert m == n, "missmatch dimension"
    Adjacance = connectivty(G)
    weights = np.sum(Adjacance, axis=1)
    if (weights == 0).any():
        raise ValueError(
            "some nodes have no edges/neighbors/connections with other nodes"
        )  # comme ça pas de problème de type sur la sortie
    transition = Eps * Adjacance * 1 / weights[:, np.newaxis] + np.identity(m)
    X_new = np.dot(transition, X) - Eps * X
    Y_new = np.dot(transition, Y) - Eps * Y
    return X_new, Y_new


def evolution_vec(G, X_0, Y_0, N, list_ab, Eps, Norm: bool, alpha=1.0):
    n, m = len(X_0), len(Y_0)
    assert n == m, "wrong dimensions, both initial conditions don't have the same size"
    X = np.zeros((n, N))
    Y = np.zeros((n, N))
    UN = np.ones(len(G.nodes))
    if Norm:
        X[:, 0] = X_0 / np.linalg.norm(X_0) / alpha
        Y[:, 0] = Y_0 / np.linalg.norm(Y_0) / alpha
        X_c, Y_c = transfo_coupling_vec(G, X[:, 0], Y[:, 0], Eps)
        for t in range(1, N):
            # print("evolution step", t)
            mediumX = -list_ab[0, :] * X_c**2 + UN + Y_c
            mediumY = list_ab[1, :] * X_c
            X[:, t] = mediumX / np.linalg.norm(mediumX) / alpha
            Y[:, t] = mediumY / np.linalg.norm(mediumY) / alpha
            X_c, Y_c = transfo_coupling_vec(G, X[:, t], Y[:, t], Eps)
        #print("compute dynamic done")
        return X, Y
    else:
        X[:, 0] = X_0
        Y[:, 0] = Y_0
        X_c, Y_c = transfo_coupling_vec(G, X[:, 0], Y[:, 0], Eps)
        for t in range(1, N):
            # print("evolution step", t)
            X[:, t] = -list_ab[0, :] * X_c**2 + UN + Y_c
            Y[:, t] = list_ab[1, :] * X_c
            X_c, Y_c = transfo_coupling_vec(G, X[:, t], Y[:, t], Eps)
        #print("compute dynamic done")
        return X, Y

there is an issue of we want to normalize againt time evolution because values are extremely small but not zero therefore if values are divided by the norm it leads to an overflow issue at some point

Exemple sur un graphe simple

Application sur le graphe de la cellule ci-dessus

In [303]:
def count2D(data, node):
    """
    warning data is either x data or y data
    """
    assert len(np.shape(data)) == 2, "wrong data shape input"
    simu = np.sort(data[node, :])
    k = len(simu)
    nb_points = 1
    start = simu[0]
    for i in range(k):
        if simu[i] != start:
            nb_points += 1
            start = simu[i]
    return nb_points

def countpdf(data, node, bins = 500):
    """
    Donne la pdf of the values taken by a given node
    """    
    assert len(np.shape(data)) == 2, "wrong data shape input"
    simu = np.sort(data[node, :])
    pdf, bins = np.histogram(simu, bins=bins, density=True)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    return bin_centers, pdf


########## CROSS_CORR + MSD





MSD







#

In [347]:
N_node = 2
std = 0.1
mean = 0.0
rng = np.random.default_rng(42)
(xmax, ymax) = (1.0, 1.0)
test_pos = buildgraph.pos_nodes_uniform(N_node, xmax, ymax, rng)
Adja = buildgraph.connexion_normal_deterministic(test_pos, rng, std, 1., 1.)
graphe = nx.from_numpy_array(Adja)

Number_Nodes = nx.number_of_nodes(graphe)


In [348]:
components = list(nx.connected_components(graphe))
representatives = [list(comp)[0] for comp in components]
for i in range(len(representatives) - 1):
    graphe.add_edge(representatives[i], representatives[i + 1])


In [349]:
fig, ax = plt.subplots()
ax.scatter(test_pos[0,:], test_pos[1, :], s=1)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Nodes positions in space")
plt.show()

In [350]:
plt.figure()
nx.draw(graphe, with_labels=True)
plt.show()

In [351]:
list_ab = np.zeros((2, Number_Nodes))
for i in range(Number_Nodes):
    list_ab[0, i] = rng.uniform(0.8, 1.4)
    list_ab[1, i] = 0.3

In [352]:
N = 2000
Epsilon = np.linspace(0, 0.5, 400)
X_0 = [0.5 for _ in range(Number_Nodes)]
Y_0 = [0.5 for _ in range(Number_Nodes)]
transitoire = 1000
node = 0

In [353]:
Xevol_forward, Yevol_forward = (
    np.zeros((len(X_0), N, len(Epsilon))),
    np.zeros((len(X_0), N, len(Epsilon))),
)
Xevol_backward, Yevol_backward = (
    np.zeros((len(X_0), N, len(Epsilon))),
    np.zeros((len(X_0), N, len(Epsilon))),
)

for i in tqdm(range(len(Epsilon))):
    Xevol_forward[:, :, i], Yevol_forward[:, :, i] = evolution_vec(
        graphe, X_0, Y_0, N, list_ab, Epsilon[i], False
    )



 26%|██▌       | 104/400 [00:05<00:16, 17.88it/s]


KeyboardInterrupt: 

In [ ]:
def MSD_xy(G, X, Y):
    n, N = X.shape
    assert n == len(G.nodes), "wrong dimension"
    MSD_values = np.zeros(N)
    for t in range(N):
        mean_X = np.mean(X[:, t])
        mean_Y = np.mean(Y[:, t])
        MSD_t = np.mean((X[:, t] - mean_X) ** 2 + (Y[:, t] - mean_Y) ** 2)
        MSD_values[t] = MSD_t
    return MSD_values

def MSD(G, X):
    n, N = X.shape
    assert n == len(G.nodes), "wrong dimension"
    MSD_values = np.zeros(N)
    for t in range(N):
        mean_X = np.mean(X[:, t])
        MSD_t = np.mean((X[:, t] - mean_X) ** 2)
        MSD_values[t] = MSD_t
    return MSD_values


In [ ]:
MSD_forward = np.zeros(len(Epsilon))
for j in range(len(Epsilon)):
        MSD_forward[j] = np.mean(MSD(graphe, Xevol_forward[:, transitoire:, j]))

MSD_backward = np.zeros(len(Epsilon))
for j in range(len(Epsilon)):
        MSD_backward[j] = np.mean(MSD(graphe, Xevol_backward[:, transitoire:, j]))

        


In [ ]:
plt.figure()
plt.scatter(Epsilon, MSD_forward, s=5, label="forward evolution")
#plt.scatter(Epsilon, MSD_backward, s=5, label="backward evolution")
plt.xlabel("Coupling constant Eps")
plt.ylabel("Mean Squared Displacement (MSD)")
plt.title("Mean Squared Displacement vs Coupling constant")
plt.legend()
plt.show()

CROSS-CORRELATION

In [358]:


def binning_data(data, num_bins):
    """Bin the data into specified number of bins and return bin centers and counts."""
    min_val = np.min(data)
    max_val = np.max(data)
    bins = np.linspace(min_val, max_val, num_bins + 1)
    counts, _ = np.histogram(data, bins=bins)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    return bin_centers, counts

def calculate_cross_correlation_at_max_lag_simple(s1: np.ndarray, s2: np.ndarray, max_lag_range: int = None):
    N = len(s1)
    if N == 0: return np.nan, np.nan
    
    if max_lag_range is None: max_lag_range = N // 2

    s1_norm = (s1 - np.mean(s1)) / (np.std(s1) + 1e-9) # Add small epsilon to avoid div by zero
    s2_norm = (s2 - np.mean(s2)) / (np.std(s2) + 1e-9)

    cross_corr = scipy.signal.correlate(s1_norm, s2_norm, mode='full') / N
    lags = np.arange(-(N - 1), N)

    valid_lags_idx = np.where((lags >= -max_lag_range) & (lags <= max_lag_range))
    lags_filtered = lags[valid_lags_idx]
    cross_corr_filtered = cross_corr[valid_lags_idx]

    if not lags_filtered.size: return np.nan, np.nan

    max_corr_abs_idx = np.argmax(np.abs(cross_corr_filtered))
    
    return cross_corr_filtered[max_corr_abs_idx], lags_filtered[max_corr_abs_idx]


def corr_between_nodes(G, X, Y, node_i, node_j, num_bins=50, max_lag=10):
    """Compute the correlation between two nodes over time."""
    n, N = X.shape
    assert n == len(G.nodes), "wrong dimension"
    data_i = X[node_i, :]
    data_j = Y[node_j, :]
    bin_centers_i, counts_i = binning_data(data_i, num_bins)
    bin_centers_j, counts_j = binning_data(data_j, num_bins)
    correlation = calculate_cross_correlation_at_max_lag_simple(counts_i, counts_j, max_lag)[0]
    return correlation

def distance_between_nodes(G, node_i, node_j):
    """Compute the shortest path distance between two nodes in the graph."""
    try:
        distance = nx.shortest_path_length(G, source=node_i, target=node_j)
    except nx.NetworkXNoPath:
        distance = np.inf  # No path exists between the nodes
    return distance

def correlation_vs_distance(G, CORR):
    n = len(G.nodes)
    distances = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            distances[i, j] = distance_between_nodes(G, i, j)
    return distances


In [359]:
Eps_index = 300 # Choose a specific Epsilon index to analyze

CORR = np.zeros((Number_Nodes, Number_Nodes))

for i in range( Number_Nodes):
    for j in range(nx.number_of_nodes(graphe)):
        CORR[i,j] = corr_between_nodes(graphe, Xevol_forward[:, transitoire:, Eps_index ], Xevol_forward[:, transitoire:, Eps_index], i, j, num_bins=50,max_lag=10)
print(CORR.shape)


DIST = correlation_vs_distance(graphe, CORR)
print(DIST.shape)



(2, 2)
(2, 2)


In [360]:

# Pour collecter les corrélations par distance
correlations_by_distance = collections.defaultdict(list)
n_nodes = CORR.shape[0]

for i in range(n_nodes):
    for j in range(i + 1, n_nodes): # Parcourir uniquement la moitié supérieure pour éviter les doublons et les auto-corrélations (distance 0)
        distance = DIST[i, j]
        correlation = CORR[i, j]
        if distance > 0: # N'inclure que les distances entre noeuds distincts
            correlations_by_distance[distance].append(correlation)

# Calculer la moyenne des corrélations pour chaque distance
avg_correlations_per_distance = {
    d: np.mean(corrs) for d, corrs in correlations_by_distance.items()
}

# Trier les distances pour le tracé
sorted_distances = sorted(avg_correlations_per_distance.keys())
average_correlations = [avg_correlations_per_distance[d] for d in sorted_distances]


In [361]:
def compute_caracteristic_distance(distances, correlations):
    """Compute the characteristic distance from correlation vs distance data."""
    distances = np.array(distances)
    correlations = np.array(correlations)

    # Normaliser les corrélations
    correlations_normalized = correlations / np.max(correlations)

    # Trouver la distance où la corrélation tombe en dessous de 1/e
    threshold = 1000
    indices_below_threshold = np.where(correlations_normalized < threshold)[0]

    if len(indices_below_threshold) == 0:
        return np.inf  # La corrélation ne tombe jamais en dessous du seuil

    characteristic_distance = distances[indices_below_threshold[0]]
    return characteristic_distance

In [362]:

# Tracer les résultats
plt.figure(figsize=(8, 6))
plt.plot(sorted_distances, average_correlations, marker='o', linestyle='-')
plt.xlabel("Distance de plus court chemin (d)")
plt.ylabel("Corrélation moyenne C(d)")
plt.xlim(left=0)
plt.ylim(0, 1)
plt.title("Corrélation moyenne vs. Distance")
plt.grid(True, linestyle='--', alpha=0.7)
# Vous pouvez essayer des échelles log si la décroissance est exponentielle
# plt.xscale('log')
# plt.yscale('log')
plt.show()

In [363]:
CARAC_L = []

for eps in tqdm(Epsilon[::10]):
    CORR = np.zeros((Number_Nodes, Number_Nodes))

    for i in range( Number_Nodes):
        for j in range(nx.number_of_nodes(graphe)):
            CORR[i,j] = corr_between_nodes(graphe, Xevol_forward[:, transitoire:, Eps_index ], Xevol_forward[:, transitoire:, Eps_index], i, j, num_bins=50,max_lag=10)
    DIST = correlation_vs_distance(graphe, CORR)

    correlations_by_distance = collections.defaultdict(list)
    n_nodes = CORR.shape[0]

    for i in range(n_nodes):
        for j in range(i + 1, n_nodes): # Parcourir uniquement la moitié supérieure pour éviter les doublons et les auto-corrélations (distance 0)
            distance = DIST[i, j]
            correlation = CORR[i, j]
            if distance > 0: # N'inclure que les distances entre noeuds distincts
                correlations_by_distance[distance].append(correlation)

    # Calculer la moyenne des corrélations pour chaque distance
    avg_correlations_per_distance = {
        d: np.mean(corrs) for d, corrs in correlations_by_distance.items()
    }

    # Trier les distances pour le tracé
    sorted_distances = sorted(avg_correlations_per_distance.keys())
    average_correlations = [avg_correlations_per_distance[d] for d in sorted_distances]
    CARAC_L.append(compute_caracteristic_distance(sorted_distances, average_correlations))
    #print(f"Eps: {eps}, L: {CARAC_L[-1]}")

plt.figure()
plt.scatter(Epsilon[::10], CARAC_L, s=5)
plt.xlabel("Coupling constant Eps")
plt.ylabel("Characteristic Distance L")
plt.title("Characteristic Distance vs Coupling constant")
plt.show()


    
    


100%|██████████| 40/40 [00:00<00:00, 1407.86it/s]
